In [1]:
#assignment 1

In [8]:
import pandas as pd
import numpy as np
import json
import requests

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

locations = {
    "Connaught Place": (28.6315, 77.2167),
    "Rohini": (28.7495, 77.0565),
    "Dwarka": (28.5921, 77.0460),
    "Saket": (28.5245, 77.2066),
    "Lajpat Nagar": (28.5677, 77.2433),
    "Karol Bagh": (28.6514, 77.1907),
    "Anand Vihar": (28.6469, 77.3160),
    "Vasant Kunj": (28.5293, 77.1541),
    "Noida": (28.5355, 77.3910),
    "Delhi Airport": (28.5562, 77.1000)
}

all_data = []

#weather
for location, (lat, lon) in locations.items():

    print("Collecting data for:", location)
    weather_params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "past_days": 30,
        "forecast_days": 0,
        "timezone": "Asia/Kolkata"
    }
    try:
      weather_response = requests.get(
        WEATHER_URL,
        params=weather_params,
        timeout=20
    )
     weather_data = weather_response.json()

     weather_df = pd.DataFrame(weather_data["hourly"])
     weather_df["time"] = pd.to_datetime(weather_df["time"])
     print("Collected", len(weather_df), "hourly records")
    except Exception as e:
      weather_df = pd.DataFrame()
       print("No internet — the next cell will create sample data.")
 

#aqi
aq_params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide",
        "past_days": 30,
        "forecast_days": 0,
        "timezone": "Asia/Kolkata"
    }

    aq_response = requests.get(
        AQ_URL,
        params=aq_params,
        timeout=20
    )

    aq_data = aq_response.json()

    aq_df = pd.DataFrame(aq_data["hourly"])

    # Convert time to datetime
    aq_df["time"] = pd.to_datetime(aq_df["time"])


merged = pd.merge(
        weather_df,
        aq_df,
        on="time",
        how="inner"
    )


    merged["location"] = location
    merged["latitude"] = lat
    merged["longitude"] = lon


    # Store this location's data
    all_data.append(merged)


    # Wait before next API request
    time.sleep(1)

all_data.shape()



IndentationError: unindent does not match any outer indentation level (<string>, line 42)

In [3]:
import requests
import pandas as pd
import time
from pathlib import Path

# --------------------------------------------------
# 1. API URLs
# --------------------------------------------------

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"


# --------------------------------------------------
# 2. Locations
# --------------------------------------------------
# latitude, longitude

locations = {
    "Connaught Place": (28.6315, 77.2167),
    "Rohini": (28.7495, 77.0565),
    "Dwarka": (28.5921, 77.0460),
    "Saket": (28.5245, 77.2066),
    "Lajpat Nagar": (28.5677, 77.2433),
    "Karol Bagh": (28.6514, 77.1907),
    "Anand Vihar": (28.6469, 77.3160),
    "Vasant Kunj": (28.5293, 77.1541),
    "Noida": (28.5355, 77.3910),
    "Delhi Airport": (28.5562, 77.1000)
}


# --------------------------------------------------
# 3. Store data from all locations
# --------------------------------------------------

all_data = []


# --------------------------------------------------
# 4. Loop through all locations
# --------------------------------------------------

for location, (lat, lon) in locations.items():

    print("Collecting data for:", location)

    # ------------------------------
    # Weather API
    # ------------------------------

    weather_params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "past_days": 30,
        "forecast_days": 0,
        "timezone": "Asia/Kolkata"
    }

    weather_response = requests.get(
        WEATHER_URL,
        params=weather_params,
        timeout=20
    )

    weather_data = weather_response.json()

    weather_df = pd.DataFrame(weather_data["hourly"])

    # Rename columns
    weather_df = weather_df.rename(columns={
        "temperature_2m": "temperature",
        "relative_humidity_2m": "humidity",
        "wind_speed_10m": "wind_speed"
    })

    # Convert time to datetime
    weather_df["time"] = pd.to_datetime(weather_df["time"])


    # ------------------------------
    # Air Quality API
    # ------------------------------

    aq_params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide",
        "past_days": 30,
        "forecast_days": 0,
        "timezone": "Asia/Kolkata"
    }

    aq_response = requests.get(
        AQ_URL,
        params=aq_params,
        timeout=20
    )

    aq_data = aq_response.json()

    aq_df = pd.DataFrame(aq_data["hourly"])

    # Convert time to datetime
    aq_df["time"] = pd.to_datetime(aq_df["time"])


    # ------------------------------
    # Merge Weather + Air Quality
    # ------------------------------

    merged = pd.merge(
        weather_df,
        aq_df,
        on="time",
        how="inner"
    )


    # ------------------------------
    # Add location information
    # ------------------------------

    merged["location"] = location
    merged["latitude"] = lat
    merged["longitude"] = lon


    # Store this location's data
    all_data.append(merged)


    # Wait before next API request
    time.sleep(1)


# --------------------------------------------------
# 5. Combine all 10 locations
# --------------------------------------------------

final_df = pd.concat(
    all_data,
    ignore_index=True
)


# --------------------------------------------------
# 6. Arrange columns
# --------------------------------------------------

final_df = final_df[
    [
        "location",
        "latitude",
        "longitude",
        "time",
        "temperature",
        "humidity",
        "precipitation",
        "wind_speed",
        "pm2_5",
        "pm10",
        "nitrogen_dioxide",
        "ozone",
        "sulphur_dioxide",
        "carbon_monoxide"
    ]
]


# --------------------------------------------------
# 7. Check the dataset
# --------------------------------------------------

print("\nDataset shape:", final_df.shape)

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nFirst 5 rows:")
display(final_df.head())


# --------------------------------------------------
# 8. Save as CSV
# --------------------------------------------------

data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

final_df.to_csv(
    data_folder / "delhi_weather_air_quality.csv",
    index=False
)

print("\nSaved successfully!")
print("File:", data_folder / "delhi_weather_air_quality.csv")


Dataset shape: (7200, 14)

Missing values:
location            0
latitude            0
longitude           0
time                0
temperature         0
humidity            0
precipitation       0
wind_speed          0
pm2_5               0
pm10                0
nitrogen_dioxide    0
ozone               0
sulphur_dioxide     0
carbon_monoxide     0
dtype: int64

First 5 rows:


,location,latitude,longitude,time,temperature,humidity,precipitation,wind_speed,pm2_5,pm10,nitrogen_dioxide,ozone,sulphur_dioxide,carbon_monoxide
0,Connaught Place,28.6315,77.2167,2026-07-17 00:00:00,34.2,61,0.0,0.5,255.2,1054.3,91.7,0.0,45.3,638.0
1,Connaught Place,28.6315,77.2167,2026-07-17 01:00:00,33.7,59,0.0,1.2,261.7,1087.8,91.8,0.0,45.9,471.0
2,Connaught Place,28.6315,77.2167,2026-07-17 02:00:00,33.2,59,0.0,0.6,267.3,1113.9,90.9,0.0,46.5,345.0
3,Connaught Place,28.6315,77.2167,2026-07-17 03:00:00,32.1,65,0.0,2.9,271.0,1113.9,90.0,0.0,48.4,279.0
4,Connaught Place,28.6315,77.2167,2026-07-17 04:00:00,31.5,71,0.0,2.7,267.3,1039.4,88.0,0.0,50.3,255.0



Saved successfully!
File: data\delhi_weather_air_quality.csv
